<div><center><img src="https://ml.ucv.ai/logo.png\" width=150"/> </center></div>

# Proyecto final
-----
# Transfer learning y fine-tuning con Hugging Face

- **Alumno 1:** NOMBRE Y APELLIDO
- **Cédula 1:** CÉDULA
- **Alumno 2:** NOMBRE Y APELLIDO
- **Cédula 2:** CÉDULA

**Modelo en el Hub:** `usuario/mi-modelo`

Este es el esqueleto del entregable. Las secciones son las del enunciado; lo que está en *cursiva* son instrucciones que deben reemplazar por su trabajo. Las celdas de código provistas (semilla, tabla de resultados, publicación) se usan tal cual.

## 0. Configuración

In [ ]:
!pip -q install transformers datasets accelerate huggingface_hub

import random, time
import numpy as np, torch
from datasets import load_dataset

CEDULA = 12345678               # su cédula de identidad: es la semilla de TODO el notebook
def fijar_semilla(seed=CEDULA):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
fijar_semilla()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| dispositivo:", device)

## 1. El problema

*Qué se clasifica, cuáles son las clases, para qué serviría el modelo y para quién. Limitaciones y consideraciones éticas de los datos. Por qué el modelo base elegido es razonable: con qué datos y para qué tarea fue preentrenado, y qué esperan que transfiera.*

## 2. Los datos

*Carga desde el Hub, tamaño de cada partición, balance de clases (gráfico), ejemplos por clase, resoluciones o longitudes, verificación de fugas entre particiones.*

In [ ]:
MODELO_BASE = "google/vit-base-patch16-224-in21k"     # su modelo base
DATASET     = "AI-Lab-Makerere/beans"                  # su dataset

ds = load_dataset(DATASET)
print(ds)

## 3. Preprocesamiento

*El `processor`/`tokenizer` del modelo base y qué hace; construcción de la partición de validación con la semilla; aumento de datos si lo usan.*

## 4. Experimentos

Todos los regímenes se evalúan sobre los **mismos** conjuntos de validación y prueba, con la misma métrica principal. Registren cada uno con `registrar(...)` para que la tabla de la sección 5 salga sola.

In [ ]:
import pandas as pd
RESULTADOS = []

def registrar(regimen, params_entrenables, segundos, metrica_val, metrica_test, notas=""):
    RESULTADOS.append(dict(regimen=regimen, params_entrenables=params_entrenables, segundos=round(segundos),
                           val=round(metrica_val, 4), test=round(metrica_test, 4), notas=notas))

def tabla_resultados():
    return pd.DataFrame(RESULTADOS).set_index("regimen")

def params_entrenables(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)

### 4.1 Baseline sin preentrenamiento

*Un modelo pequeño desde cero sobre sus datos.*

### 4.2 Modelo base sin adaptar (si aplica)

*Zero-shot o cabeza compatible. Si no aplica, explíquenlo.*

### 4.3 Feature extraction

*Backbone congelado (`requires_grad = False` en `model.base_model`), solo la cabeza nueva.*

### 4.4 Fine-tuning completo

*Todo descongelado, tasa de aprendizaje baja; justifiquen tasa, épocas y criterio de parada con validación.*

### 4.5 Experimento adicional (cuenta para α)

*Eficiencia de datos, descongelado gradual, dos modelos base, catastrophic forgetting...*

## 5. Resultados y discusión

*Tabla, curvas, matriz de confusión del mejor modelo, errores típicos, y la discusión: qué ganó, por qué, cuánto costó, qué harían con más datos o cómputo.*

In [ ]:
tabla_resultados()

## 6. Modelo óptimo y publicación

*Criterio de optimalidad por escrito. Luego, publicar el modelo elegido y comprobar que se recarga tal como lo hará el API.*

In [ ]:
from huggingface_hub import login
login()                          # token con permiso de escritura (huggingface.co/settings/tokens)

REPO_ID = "usuario/mi-modelo"    # el mismo identificador que va en el .env del API
mejor_modelo.push_to_hub(REPO_ID)
processor.push_to_hub(REPO_ID)   # o tokenizer.push_to_hub(REPO_ID)

In [ ]:
# Prueba de recarga desde el Hub: exactamente lo que hace el API al arrancar
from transformers import pipeline
clf = pipeline("image-classification", model=REPO_ID, top_k=None)   # o "text-classification"
clf(ds["test"][0]["image"])                                         # o clf("un texto de prueba")

## 7. Conclusiones y limitaciones

*Qué aprendieron, dónde falla el modelo, sesgos de los datos, uso previsto. Este texto es también la base de la model card.*